# FractalTrainer — Sprint 19 scale stress test (N=1000 experts)

Addresses paper §5 limitation: *"Scale untested. The registry tops out at N=15 in the main experiments [...] Behavior at N=1000+ (realistic deployment scale) is not validated."*

### What this notebook measures

Builds 1000 distinct MNIST-binary-subset experts, registers their 1000-d softmax signatures, and at N ∈ {15, 50, 100, 250, 500, 1000} reports:

1. **Routing accuracy (LOO)** — hold out one expert, re-compute its signature on fresh data from the same subset, test whether k-NN routing returns another expert trained on the same subset. This is the "does routing still work at scale?" check.
2. **Routing latency** — wall-time per nearest-neighbor query with a linear scan.
3. **Within-task vs cross-task distance gap** — does the clean gap from N=15 (within 3.0 / cross 10.7) survive?
4. **Match/compose/spawn verdict distribution** — at a fixed threshold calibrated from N=15, how do verdicts change as N grows?

Pass criteria (pre-registered):
- Routing accuracy ≥ 90% at N=1000
- Per-query latency < 50 ms at N=1000 on CPU (linear scan should stay trivial)
- Within / cross distance distributions remain separable (gap > 0)

### Runtime

- **GPU (T4):** ~35–50 min.
- **CPU:** ~2 hours.

### Prerequisites

Latest FractalTrainer master must be pushed to GitHub. If local master is ahead of origin/master, run `git push origin master` in the repo first.

## 1. Environment + clone repo

In [ ]:
import os, sys, subprocess, time

REPO_URL = "https://github.com/vegarjr/FractalTrainer.git"
REPO_DIR = "/content/FractalTrainer"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
print("repo at", REPO_DIR)
print("HEAD:", subprocess.check_output(["git", "-C", REPO_DIR, "log", "--oneline", "-1"]).decode().strip())

In [ ]:
import torch
print("torch", torch.__version__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Load MNIST + fixed probe batch

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms

MNIST_ROOT = "/content/mnist_data"

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_ds = datasets.MNIST(MNIST_ROOT, train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(MNIST_ROOT, train=False, download=True, transform=transform)

train_labels = np.array(train_ds.targets)
test_labels  = np.array(test_ds.targets)

# Fixed 100-image probe — must be identical for every expert's signature
probe_rng = np.random.default_rng(0)
probe_idx = probe_rng.choice(len(test_ds), size=100, replace=False)
probe_X = torch.stack([test_ds[i][0] for i in probe_idx]).view(100, -1)  # (100, 784)
print("probe shape:", probe_X.shape)

## 3. Task catalogue — 500 subsets × 2 seeds = 1000 experts

For each subset S ⊂ {0..9} we keep only MNIST examples whose label ∈ S. All k-subsets of {0..9} for k ∈ {3, 4, 5, 6, 7} give C(10,3)+C(10,4)+C(10,5)+C(10,6)+C(10,7) = 120+210+252+210+120 = **912** distinct subsets. Drop to 500 by deterministic sampling, then × 2 training seeds = 1000 experts.

In [ ]:
from itertools import combinations

all_subsets = []
for k in (3, 4, 5, 6, 7):
    for combo in combinations(range(10), k):
        all_subsets.append(combo)
print(f"distinct k-subsets, k in 3..7: {len(all_subsets)}")

rng = np.random.default_rng(42)
perm = rng.permutation(len(all_subsets))
chosen = [all_subsets[i] for i in perm[:500]]

SEEDS = [42, 101]
TASKS = [(subset, seed) for subset in chosen for seed in SEEDS]
print(f"total experts to train: {len(TASKS)}")

## 4. Per-task data loader + training helper

In [ ]:
from fractaltrainer.integration.context_mlp import ContextAwareMLP
from fractaltrainer.integration.signatures import softmax_signature
from fractaltrainer.registry import FractalEntry, FractalRegistry
import torch.nn.functional as F


class _FlatSubset(Dataset):
    """Flattens MNIST images to 784-d and filters to a label subset."""
    def __init__(self, base, subset, train=True):
        self.base = base
        self.idx = np.where(np.isin(np.array(base.targets), list(subset)))[0]
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        x, y = self.base[int(self.idx[i])]
        return x.view(-1), int(y)


def train_one_expert(subset, seed, n_steps=200, lr=0.01, batch_size=64,
                     device=DEVICE):
    """Train a fresh ContextAwareMLP on MNIST filtered to `subset`,
    return its softmax signature (1000-d) and training accuracy."""
    torch.manual_seed(seed); np.random.seed(seed)
    ds = _FlatSubset(train_ds, subset)
    if len(ds) == 0:
        raise ValueError("empty subset")
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True,
                        num_workers=0, drop_last=False)
    model = ContextAwareMLP(context_scale=0.0).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    step = 0
    iterator = iter(loader)
    while step < n_steps:
        try:
            x, y = next(iterator)
        except StopIteration:
            iterator = iter(loader)
            x, y = next(iterator)
        x = x.to(device); y = y.to(device)
        opt.zero_grad()
        logits = model(x, context=None)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        opt.step()
        step += 1
    model.eval()
    probe = probe_X.to(device)
    sig = softmax_signature(model, probe)
    # Quick train accuracy on a small sample
    with torch.no_grad():
        sx, sy = next(iter(DataLoader(ds, batch_size=256, shuffle=True)))
        sx = sx.to(device); sy = sy.to(device)
        pred = model(sx, context=None).argmax(dim=1)
        acc = float((pred == sy).float().mean().item())
    del model; torch.cuda.empty_cache() if device.type == "cuda" else None
    return sig, acc


# Smoke test — one expert
t = time.time()
sig0, acc0 = train_one_expert((0, 1, 2), 42, n_steps=50)
print(f"smoke: subset=(0,1,2) seed=42  acc={acc0:.3f}  sig_l2={np.linalg.norm(sig0):.3f}  [{time.time()-t:.1f}s]")

## 5. Train all 1000 experts + collect FractalEntries

Streaming: train → signature → register → discard model. Memory stays constant.

In [ ]:
N_STEPS_PER_EXPERT = 200

registry_full = FractalRegistry()
entry_meta = []  # parallel list for analysis

t0 = time.time()
for i, (subset, seed) in enumerate(TASKS):
    sig, acc = train_one_expert(subset, seed, n_steps=N_STEPS_PER_EXPERT)
    task_name = "_".join(str(d) for d in subset)
    entry_name = f"{task_name}__s{seed}"
    meta = {"task": task_name, "subset": list(subset), "seed": seed,
            "train_acc": acc}
    entry = FractalEntry(name=entry_name, signature=sig, metadata=meta)
    registry_full.add(entry)
    entry_meta.append(meta)
    if (i + 1) % 50 == 0:
        dt = time.time() - t0
        eta = dt / (i + 1) * (len(TASKS) - i - 1)
        print(f"  {i+1:>4d}/{len(TASKS)}  elapsed={dt:.0f}s  eta={eta:.0f}s  "
              f"last_acc={acc:.3f}")

print(f"\nAll experts trained. Wall time: {time.time()-t0:.0f}s.")
print(f"Registry size: {len(registry_full)}")
print(f"Mean train acc: {np.mean([m['train_acc'] for m in entry_meta]):.3f}")

## 6. Distance distributions at N ∈ {15, 50, 100, 250, 500, 1000}

In [ ]:
N_GRID = [15, 50, 100, 250, 500, 1000]

entries_all = registry_full.entries()  # list in insertion order

def subsample_registry(entries, N, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(entries))[:N]
    sub = FractalRegistry()
    for i in idx:
        sub.add(entries[i])
    return sub


def within_cross_distances(reg):
    """Compute within-task and cross-task pairwise L2 distances.
    Within-task = same `subset` tuple; Cross-task = different subset."""
    ents = reg.entries()
    sigs = np.stack([e.signature for e in ents])
    tasks = [tuple(e.metadata["subset"]) for e in ents]
    n = len(ents)
    # Pairwise L2
    D = np.linalg.norm(sigs[:, None, :] - sigs[None, :, :], axis=-1)
    within, cross = [], []
    for i in range(n):
        for j in range(i + 1, n):
            (within if tasks[i] == tasks[j] else cross).append(D[i, j])
    return np.array(within), np.array(cross)


distance_stats = {}
for N in N_GRID:
    reg = subsample_registry(entries_all, N, seed=0)
    w, c = within_cross_distances(reg)
    distance_stats[N] = {
        "n_within_pairs": int(w.size),
        "n_cross_pairs": int(c.size),
        "within_mean": float(w.mean()) if w.size else None,
        "within_std":  float(w.std())  if w.size else None,
        "cross_mean":  float(c.mean()),
        "cross_std":   float(c.std()),
        "gap": float(c.mean() - (w.mean() if w.size else 0.0)),
    }
    w_str = f"{w.mean():.3f}" if w.size else "n/a"
    print(f"  N={N:>5d}  within={w_str:>8s} ({w.size:>4d} pairs)  "
          f"cross={c.mean():.3f}  gap={distance_stats[N]['gap']:+.3f}")

## 7. Routing accuracy (LOO, top-k) at each N

For each entry in the subsampled registry: remove it, train a *fresh* expert on the same subset with a new seed, compute its signature, and check whether the top-1 nearest neighbor in the registry has the same subset. This is the operational "does routing work?" test.

In [ ]:
# Pre-compute 'held-out' signatures: for each distinct subset, train ONE more
# expert with a third seed and use it as the query signature.
HOLD_OUT_SEED = 9001
subset_to_heldout_sig = {}

t0 = time.time()
unique_subsets = sorted({tuple(m["subset"]) for m in entry_meta})
print(f"Training {len(unique_subsets)} held-out experts (seed={HOLD_OUT_SEED}) for routing test...")
for i, subset in enumerate(unique_subsets):
    sig, _ = train_one_expert(subset, HOLD_OUT_SEED, n_steps=N_STEPS_PER_EXPERT)
    subset_to_heldout_sig[subset] = sig
    if (i + 1) % 50 == 0:
        dt = time.time() - t0
        print(f"  {i+1}/{len(unique_subsets)}  elapsed={dt:.0f}s")
print(f"Held-out experts done, {time.time()-t0:.0f}s.")

In [ ]:
routing_stats = {}
for N in N_GRID:
    reg = subsample_registry(entries_all, N, seed=0)
    ents = reg.entries()
    reg_subsets = {tuple(e.metadata["subset"]) for e in ents}
    # For each unique subset represented in the subsampled registry,
    # use its held-out signature as a query.
    queries = [(s, subset_to_heldout_sig[s]) for s in reg_subsets
               if s in subset_to_heldout_sig]
    top1_correct = 0
    top3_correct = 0
    latencies = []
    for subset, qsig in queries:
        t = time.time()
        res = reg.find_nearest(qsig, k=3, query_name="_".join(str(d) for d in subset))
        latencies.append((time.time() - t) * 1000.0)
        hit_subsets = [tuple(e.metadata["subset"]) for e in res.entries]
        if hit_subsets and hit_subsets[0] == subset:
            top1_correct += 1
        if subset in hit_subsets:
            top3_correct += 1
    routing_stats[N] = {
        "n_queries": len(queries),
        "top1_acc": top1_correct / max(1, len(queries)),
        "top3_acc": top3_correct / max(1, len(queries)),
        "latency_ms_mean": float(np.mean(latencies)),
        "latency_ms_p95":  float(np.percentile(latencies, 95)),
    }
    print(f"  N={N:>5d}  top1={routing_stats[N]['top1_acc']:.3f}  "
          f"top3={routing_stats[N]['top3_acc']:.3f}  "
          f"latency={routing_stats[N]['latency_ms_mean']:.2f} ms  "
          f"(p95 {routing_stats[N]['latency_ms_p95']:.2f} ms)")

## 8. Verdict distribution at fixed thresholds (calibrated from N=15)

In [ ]:
# Calibrate thresholds from the smallest (N=15) registry — this is what
# a practitioner would do in practice.
reg15 = subsample_registry(entries_all, 15, seed=0)
cal = reg15.calibrate_thresholds(task_key="task", within_percentile=95.0,
                                  cross_percentile=5.0)
MATCH_T = float(cal.match_threshold)
SPAWN_T = float(cal.spawn_threshold)
print(f"Thresholds from N=15: match={MATCH_T:.3f}  spawn={SPAWN_T:.3f}")

verdict_stats = {}
for N in N_GRID:
    reg = subsample_registry(entries_all, N, seed=0)
    ents = reg.entries()
    reg_subsets = {tuple(e.metadata["subset"]) for e in ents}
    # In-subset queries: should verdict=match ideally
    in_verdicts = {"match": 0, "compose": 0, "spawn": 0, "empty": 0}
    for subset in reg_subsets:
        if subset not in subset_to_heldout_sig: continue
        d = reg.decide(subset_to_heldout_sig[subset], MATCH_T, SPAWN_T)
        in_verdicts[d.verdict] += 1
    # Out-subset queries: held-outs whose subset is NOT in this registry
    out_verdicts = {"match": 0, "compose": 0, "spawn": 0, "empty": 0}
    for subset, qsig in subset_to_heldout_sig.items():
        if subset in reg_subsets: continue
        d = reg.decide(qsig, MATCH_T, SPAWN_T)
        out_verdicts[d.verdict] += 1
    total_in = max(1, sum(in_verdicts.values()))
    total_out = max(1, sum(out_verdicts.values()))
    verdict_stats[N] = {
        "match_threshold": MATCH_T, "spawn_threshold": SPAWN_T,
        "in_counts": in_verdicts,
        "in_fractions": {k: v / total_in for k, v in in_verdicts.items()},
        "out_counts": out_verdicts,
        "out_fractions": {k: v / total_out for k, v in out_verdicts.items()},
    }
    print(f"  N={N:>5d}  in-subset: match={in_verdicts['match']/total_in:.2f} "
          f"compose={in_verdicts['compose']/total_in:.2f} "
          f"spawn={in_verdicts['spawn']/total_in:.2f}")
    print(f"           out-subset: match={out_verdicts['match']/total_out:.2f} "
          f"compose={out_verdicts['compose']/total_out:.2f} "
          f"spawn={out_verdicts['spawn']/total_out:.2f}")

## 9. Plots

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# (a) routing accuracy vs N
ax = axes[0, 0]
Ns = np.array(N_GRID)
ax.plot(Ns, [routing_stats[N]["top1_acc"] for N in N_GRID], "o-", label="top-1")
ax.plot(Ns, [routing_stats[N]["top3_acc"] for N in N_GRID], "s--", label="top-3")
ax.axhline(0.9, color="gray", ls=":", alpha=0.5, label="pass gate (0.9)")
ax.set_xscale("log"); ax.set_xlabel("Registry size N")
ax.set_ylabel("Routing accuracy"); ax.set_ylim(0, 1.02)
ax.set_title("Routing accuracy vs registry size")
ax.legend(); ax.grid(alpha=0.3)

# (b) routing latency vs N
ax = axes[0, 1]
ax.plot(Ns, [routing_stats[N]["latency_ms_mean"] for N in N_GRID], "o-", label="mean")
ax.plot(Ns, [routing_stats[N]["latency_ms_p95"]  for N in N_GRID], "s--", label="p95")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("Registry size N")
ax.set_ylabel("Query latency (ms)"); ax.set_title("Linear-scan routing latency vs N")
ax.legend(); ax.grid(alpha=0.3)

# (c) within/cross distance gap vs N
ax = axes[1, 0]
w_means = [distance_stats[N]["within_mean"] for N in N_GRID]
c_means = [distance_stats[N]["cross_mean"]  for N in N_GRID]
w_stds  = [distance_stats[N]["within_std"]  for N in N_GRID]
c_stds  = [distance_stats[N]["cross_std"]   for N in N_GRID]
# Replace None with nan
w_means = [np.nan if v is None else v for v in w_means]
w_stds  = [np.nan if v is None else v for v in w_stds]
ax.errorbar(Ns, c_means, yerr=c_stds, fmt="s-", label="cross-task", capsize=3)
ax.errorbar(Ns, w_means, yerr=w_stds, fmt="o-", label="within-task", capsize=3)
ax.set_xscale("log"); ax.set_xlabel("Registry size N")
ax.set_ylabel("L2 signature distance"); ax.set_title("Within-task vs cross-task distance")
ax.legend(); ax.grid(alpha=0.3)

# (d) verdict distribution vs N
ax = axes[1, 1]
for verdict in ["match", "compose", "spawn"]:
    in_fracs = [verdict_stats[N]["in_fractions"][verdict] for N in N_GRID]
    ax.plot(Ns, in_fracs, "o-", label=f"in-subset {verdict}")
ax.set_xscale("log"); ax.set_xlabel("Registry size N"); ax.set_ylabel("Fraction of queries")
ax.set_ylim(0, 1.02); ax.set_title(f"Verdict distribution (in-subset queries)\n"
                                     f"match_t={MATCH_T:.2f}  spawn_t={SPAWN_T:.2f}")
ax.legend(); ax.grid(alpha=0.3)

fig.tight_layout()
out_path = "/content/sprint19_scale_n1000.png"
fig.savefig(out_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"plot saved: {out_path}")

## 10. Save results JSON

In [ ]:
import json

payload = {
    "config": {
        "n_experts": len(TASKS),
        "n_unique_subsets": len(unique_subsets),
        "seeds_per_subset": len(SEEDS),
        "n_steps_per_expert": N_STEPS_PER_EXPERT,
        "probe_size": int(probe_X.shape[0]),
        "signature_dim": int(probe_X.shape[0]) * 10,
        "match_threshold": MATCH_T,
        "spawn_threshold": SPAWN_T,
        "device": str(DEVICE),
    },
    "distance_stats": distance_stats,
    "routing_stats":  routing_stats,
    "verdict_stats":  verdict_stats,
}
out_json = "/content/sprint19_scale_n1000.json"
with open(out_json, "w") as fh:
    json.dump(payload, fh, indent=2, default=float)
print(f"results saved: {out_json}")

print("\n=== Pre-registered gates ===")
top1_at_1000 = routing_stats[1000]["top1_acc"]
lat_at_1000  = routing_stats[1000]["latency_ms_p95"]
gap_at_1000  = distance_stats[1000]["gap"]
print(f"top-1 routing at N=1000:  {top1_at_1000:.3f}   {'PASS' if top1_at_1000 >= 0.9 else 'FAIL'}  (threshold 0.9)")
print(f"p95 latency at N=1000:    {lat_at_1000:.2f} ms   {'PASS' if lat_at_1000 < 50.0 else 'FAIL'}  (threshold 50 ms)")
print(f"within/cross gap at N=1000: {gap_at_1000:+.3f}   {'PASS' if gap_at_1000 > 0.0 else 'FAIL'}  (threshold > 0)")

## 11. Download results

Run this cell to download `sprint19_scale_n1000.{json,png}` to your local machine so they can be committed back to the repo.

In [ ]:
try:
    from google.colab import files
    files.download(out_json)
    files.download(out_path)
except ImportError:
    print(f"Not in Colab; files at:\n  {out_json}\n  {out_path}")